# **SESIÓN 3:** Autocorrelación, Calidad de Datos y Modelos Baseline
## Universidad Autónoma de Occidente
### Maestría en Inteligencia Artificial y Ciencias de Datos

**Objetivo del notebook:** preparar una serie temporal semanal de casos de dengue, revisar su calidad, construir modelos baseline, evaluar sus errores y diagnosticar si todavía quedan patrones temporales en los residuos.

> Esta versión conserva el estilo del notebook original, pero reorganiza el flujo para evitar fugas de información y reutilizar código mediante funciones simples.

---
## ⚙️ PARTE 1: Configuración del Entorno

En esta sección se instalan y cargan las librerías necesarias para el análisis. Se dejan definidas las rutas, constantes y parámetros principales del experimento.

In [ ]:
!pip install statsforecast utilsforecast statsmodels -q

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, WindowAverage, RandomWalkWithDrift

from utilsforecast.losses import mae, rmse, smape
from utilsforecast.evaluation import evaluate

import warnings
warnings.filterwarnings('ignore')

# =========================
# Parámetros principales
# =========================
RUTA_DATOS_CRUDOS = "/content/datos_dengue_202604282143.csv"
RUTA_SALIDA_NIXTLA = "/content/dengue_anio_semana_nixtla.csv"

UNIQUE_ID = "dengue_cali"
COLUMNA_FECHA = "fec_not"
FRECUENCIA = "W-MON"
FECHA_CORTE = "2020-01-01"
ESTACIONALIDAD_SEMANAL = 52

MODELOS_BASELINE = ["Naive", "SeasonalNaive", "WindowAverage", "RWD"]

print("✅ Todas las librerías cargadas correctamente")

---
## 🧰 PARTE 1.1: Funciones Reutilizables

Para mantener el notebook ordenado, las operaciones repetidas se encapsulan en funciones. La idea es conservar el procedimiento original, pero hacerlo más limpio y fácil de reutilizar.

In [ ]:
def leer_datos_dengue(ruta_archivo):
    """Carga el archivo original de dengue."""
    datos = pd.read_csv(ruta_archivo)
    print("Primeras filas del dataset original:")
    display(datos.head())
    print("\nColumnas disponibles:")
    print(datos.columns.tolist())
    return datos


def construir_serie_semanal_nixtla(datos, columna_fecha="fec_not", unique_id="dengue_cali"):
    """
    Convierte el dataset diario/individual en una serie semanal con formato Nixtla:
    unique_id, ds, y.
    """
    datos = datos.copy()
    datos[columna_fecha] = pd.to_datetime(datos[columna_fecha], errors="coerce")
    datos = datos.dropna(subset=[columna_fecha])

    calendario_iso = datos[columna_fecha].dt.isocalendar()
    datos["anio"] = calendario_iso.year.astype(int)
    datos["semana"] = calendario_iso.week.astype(int)

    # Lunes de cada semana ISO. Este formato evita ambigüedades entre año calendario y año ISO.
    datos["ds"] = pd.to_datetime(
        datos["anio"].astype(str) + "-W" + datos["semana"].astype(str).str.zfill(2) + "-1",
        format="%G-W%V-%u"
    )

    serie = (
        datos.groupby("ds")
        .size()
        .reset_index(name="y")
        .sort_values("ds")
        .reset_index(drop=True)
    )

    serie.insert(0, "unique_id", unique_id)
    serie = serie[["unique_id", "ds", "y"]]
    return serie


def guardar_serie_nixtla(serie, ruta_salida):
    """Guarda la serie semanal en formato Nixtla."""
    serie.to_csv(ruta_salida, index=False)
    print("Archivo generado correctamente:")
    print(ruta_salida)


def graficar_serie(serie, titulo="Número de Casos de Dengue por Semana"):
    """Grafica la serie temporal completa."""
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=serie["ds"],
            y=serie["y"],
            mode="lines+markers",
            name="Casos de Dengue",
            line=dict(color="#2196F3", width=1.8),
            marker=dict(size=4)
        )
    )
    fig.update_layout(
        title=titulo,
        xaxis_title="Semana Epidemiológica",
        yaxis_title="Número de Casos",
        height=450,
        template="plotly_white"
    )
    fig.show()


def completar_calendario_semanal(serie, frecuencia="W-MON", unique_id="dengue_cali"):
    """Crea el calendario semanal completo y deja NaN donde falten semanas."""
    rango_completo = pd.date_range(serie.ds.min(), serie.ds.max(), freq=frecuencia)
    calendario = pd.DataFrame({"ds": rango_completo, "unique_id": unique_id})
    serie_completa = calendario.merge(serie, on=["unique_id", "ds"], how="left")
    return serie_completa[["unique_id", "ds", "y"]]


def diagnosticar_faltantes(serie, frecuencia="W-MON"):
    """Revisa continuidad temporal y valores faltantes en y."""
    fechas_esperadas = pd.date_range(serie.ds.min(), serie.ds.max(), freq=frecuencia)
    fechas_faltantes = fechas_esperadas.difference(serie.ds)

    print("=" * 55)
    print("  DIAGNÓSTICO DE VALORES FALTANTES")
    print("=" * 55)
    print(f"  Observaciones esperadas: {len(fechas_esperadas)}")
    print(f"  Observaciones presentes: {len(serie)}")
    print(f"  Fechas faltantes:        {len(fechas_faltantes)}")
    print(f"  NaN en columna y:        {serie.y.isna().sum()}")

    if len(fechas_faltantes) == 0 and serie.y.isna().sum() == 0:
        print("\n  ✅ Serie completa — sin valores faltantes")
    else:
        print(f"\n  ⚠️  Fechas faltantes: {fechas_faltantes.tolist()}")

    return fechas_faltantes


def separar_train_test(serie, fecha_corte):
    """Hace split temporal. Nunca se aleatoriza una serie temporal."""
    fecha_corte = pd.to_datetime(fecha_corte)
    train = serie[serie.ds < fecha_corte].copy()
    test = serie[serie.ds >= fecha_corte].copy()
    return train, test


def graficar_split(train, test, fecha_corte):
    """Grafica el conjunto de entrenamiento y prueba."""
    proporcion_train = len(train) / (len(train) + len(test)) * 100
    fecha_corte = pd.to_datetime(fecha_corte)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=train.ds, y=train.y, mode="lines+markers",
        name="Train", line=dict(color="#2196F3", width=2), marker=dict(size=3)
    ))
    fig.add_trace(go.Scatter(
        x=test.ds, y=test.y, mode="lines+markers",
        name="Test", line=dict(color="#FF5722", width=2), marker=dict(size=3)
    ))
    fig.add_shape(
        type="line", xref="x", yref="paper",
        x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
        line=dict(dash="dash", color="gray", width=1.5)
    )
    fig.add_annotation(
        x=fecha_corte, y=1.02, yref="paper",
        text="Corte", showarrow=False,
        font=dict(size=10, color="gray"), xanchor="left"
    )
    fig.update_layout(
        title=f"Train-Test Split Temporal — {proporcion_train:.0f}% / {100 - proporcion_train:.0f}%",
        xaxis_title="Fecha", yaxis_title="Casos",
        height=380, template="plotly_white"
    )
    fig.show()


def calcular_medias_estacionales(train):
    """Calcula la media por semana del año usando únicamente el train."""
    train_aux = train.copy()
    train_aux["semana_anio"] = train_aux.ds.dt.isocalendar().week.astype(int)
    medias = train_aux.groupby("semana_anio")["y"].mean().to_dict()
    media_global = train_aux["y"].mean()
    return medias, media_global


def imputar_por_media_estacional(datos, medias_estacionales, media_global):
    """Imputa NaN usando la media estacional calculada previamente."""
    datos = datos.copy()
    datos["semana_anio"] = datos.ds.dt.isocalendar().week.astype(int)

    def imputar_fila(fila):
        if pd.isna(fila["y"]):
            return medias_estacionales.get(fila["semana_anio"], media_global)
        return fila["y"]

    datos["y"] = datos.apply(imputar_fila, axis=1)
    datos = datos.drop(columns=["semana_anio"])
    return datos

In [ ]:
def detectar_outliers_iqr(serie):
    """Detecta outliers usando el método IQR global."""
    q1, q3 = serie.y.quantile(0.25), serie.y.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    outliers = serie[(serie.y < limite_inf) | (serie.y > limite_sup)].copy()
    return outliers, q1, q3, iqr, limite_inf, limite_sup


def detectar_outliers_zscore(serie, umbral=3):
    """Detecta outliers usando Z-score."""
    z_scores = np.abs(stats.zscore(serie.y))
    return serie[z_scores > umbral].copy()


def graficar_outliers_iqr(serie, outliers, limite_inf, limite_sup):
    """Grafica la serie con límites IQR."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=serie.ds, y=serie.y, mode="lines",
        name="Serie", line=dict(color="#2196F3", width=1.5)
    ))
    fig.add_hline(y=limite_sup, line_dash="dash", line_color="orange", annotation_text="Límite IQR superior")
    fig.add_hline(y=limite_inf, line_dash="dash", line_color="orange", annotation_text="Límite IQR inferior")

    if len(outliers):
        fig.add_trace(go.Scatter(
            x=outliers.ds, y=outliers.y,
            mode="markers", name="Outlier (IQR)",
            marker=dict(color="red", size=10, symbol="circle-open", line_width=2)
        ))

    fig.update_layout(
        title="Detección de Outliers — Método IQR",
        height=380, template="plotly_white",
        xaxis_title="Fecha", yaxis_title="Casos"
    )
    fig.show()


def detectar_outliers_iqr_por_mes(serie):
    """Detecta outliers aplicando IQR dentro de cada mes."""
    df_check = serie.copy()
    df_check["mes"] = df_check.ds.dt.month
    df_check["outlier_mensual"] = False

    for mes in range(1, 13):
        mascara = df_check.mes == mes
        valores_mes = df_check.loc[mascara, "y"]
        q1, q3 = valores_mes.quantile(0.25), valores_mes.quantile(0.75)
        iqr_mes = q3 - q1
        es_outlier = (valores_mes < q1 - 1.5 * iqr_mes) | (valores_mes > q3 + 1.5 * iqr_mes)
        df_check.loc[mascara & es_outlier, "outlier_mensual"] = True

    return df_check


def graficar_cambios_regimen(serie, ventana=2):
    """Grafica media y desviación estándar móvil."""
    media_movil = serie.y.rolling(ventana, center=True).mean()
    std_movil = serie.y.rolling(ventana, center=True).std()

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f"Serie + Media Móvil (ventana={ventana})",
            "Desv. Estándar Móvil — cambios indican heteroscedasticidad"
        ],
        vertical_spacing=0.15
    )

    fig.add_trace(go.Scatter(
        x=serie.ds, y=serie.y, mode="lines", name="Original",
        line=dict(color="lightblue", width=1.5)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=serie.ds, y=media_movil, mode="lines", name="Media móvil",
        line=dict(color="#2196F3", width=2.5)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=serie.ds, y=std_movil, mode="lines", name="Desv. estándar",
        line=dict(color="#FF5722", width=2), fill="tozeroy",
        fillcolor="rgba(255,87,34,0.1)"
    ), row=2, col=1)

    fig.update_layout(
        height=550, template="plotly_white",
        title="Detección de Cambios de Régimen — Estadísticas Móviles"
    )
    fig.show()


def evaluar_cambio_regimen(serie):
    """Compara dos subperíodos usando Levene y t-test."""
    n_total = len(serie)
    mitad = n_total // 2
    y1 = serie.y.values[:mitad]
    y2 = serie.y.values[mitad:]

    stat_var, p_var = stats.levene(y1, y2)
    stat_med, p_med = stats.ttest_ind(y1, y2)

    print("  TEST DE CAMBIO DE RÉGIMEN — Comparación de dos subperíodos")
    print(f"  Período 1: {serie.ds.iloc[0].strftime('%Y-%m')} → {serie.ds.iloc[mitad - 1].strftime('%Y-%m')}")
    print(f"  Período 2: {serie.ds.iloc[mitad].strftime('%Y-%m')} → {serie.ds.iloc[-1].strftime('%Y-%m')}")
    print()
    print(f"  Media P1={y1.mean():.0f}  |  Media P2={y2.mean():.0f}")
    print(f"  Desv. P1={y1.std():.0f}   |  Desv. P2={y2.std():.0f}")
    print()
    print(f"  Test Levene (varianzas iguales): stat={stat_var:.3f}  p={p_var:.4f}",
          "→ Varianzas distintas ⚠️" if p_var < 0.05 else "→ Varianzas similares ✅")
    print(f"  Test t (medias iguales):         stat={stat_med:.3f}  p={p_med:.4f}",
          "→ Medias distintas ⚠️" if p_med < 0.05 else "→ Medias similares ✅")
    print()
    print("  Interpretación:")
    if p_var < 0.05 or p_med < 0.05:
        print("  ⚠️  Hay evidencia de cambio estructural entre períodos.")
        print("     Estrategias: dummy variable, segmentar la serie, o modelar con SARIMA.")
    else:
        print("  ✅  Los dos subperíodos son estadísticamente similares.")
        print("     La serie no muestra cambios de régimen significativos.")

    return {"p_varianza": p_var, "p_media": p_med}

In [ ]:
def crear_statsforecast_baseline(frecuencia="W-MON", estacionalidad=52):
    """Crea el objeto StatsForecast con los modelos baseline del taller."""
    return StatsForecast(
        models=[
            Naive(),
            SeasonalNaive(season_length=estacionalidad),
            WindowAverage(window_size=3),
            RandomWalkWithDrift()
        ],
        freq=frecuencia
    )


def entrenar_y_predecir_baselines(train, test, frecuencia="W-MON", estacionalidad=52):
    """Entrena los baselines y predice el horizonte del test."""
    horizonte = len(test)
    sf = crear_statsforecast_baseline(frecuencia=frecuencia, estacionalidad=estacionalidad)
    sf.fit(train)
    preds = sf.predict(h=horizonte)

    print("Pronósticos generados:")
    print(f"  Modelos: {[c for c in preds.columns if c not in ['unique_id', 'ds']]}")
    print(f"  Horizonte: {horizonte} semanas")
    display(preds.head())
    return sf, preds


def unir_predicciones_con_test(test, preds):
    """Une los valores reales del test con las predicciones."""
    return test.merge(preds, on=["unique_id", "ds"])


def graficar_pronosticos_baseline(serie, test, test_preds, fecha_corte, modelos_col):
    """Grafica histórico, test y pronósticos de cada baseline."""
    colores = ["#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]
    fecha_corte = pd.to_datetime(fecha_corte)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=serie.ds, y=serie.y, mode="lines", name="Histórico",
        line=dict(color="lightgray", width=1)
    ))
    fig.add_trace(go.Scatter(
        x=test.ds, y=test.y, mode="lines+markers", name="Real (test)",
        line=dict(color="black", width=1), marker=dict(size=2)
    ))

    for modelo, color in zip(modelos_col, colores):
        fig.add_trace(go.Scatter(
            x=test_preds.ds, y=test_preds[modelo], mode="lines+markers", name=modelo,
            line=dict(color=color, width=1, dash="dot"), marker=dict(size=2)
        ))

    fig.add_shape(
        type="line", xref="x", yref="paper",
        x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
        line=dict(dash="dash", color="gray", width=1)
    )
    fig.add_annotation(
        x=fecha_corte, y=1.02, yref="paper",
        text="Inicio test", showarrow=False,
        font=dict(size=10, color="gray"), xanchor="left"
    )
    fig.update_layout(
        title="Comparación de Baselines — Casos Dengue",
        xaxis_title="Fecha", yaxis_title="Casos",
        height=480, template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.show()


def calcular_metricas_baseline(train, test_preds, modelos_col):
    """Calcula MAE, RMSE, sMAPE y MASE para los modelos baseline."""
    naive_train_mae = float(np.abs(np.diff(train.y.values)).mean())

    evaluacion = evaluate(
        test_preds,
        metrics=[mae, rmse, smape],
        models=modelos_col,
        target_col="y"
    )

    fila_mae = evaluacion[evaluacion.metric == "mae"].copy()
    fila_mase = fila_mae.copy()
    fila_mase["metric"] = "mase"

    for modelo in modelos_col:
        fila_mase[modelo] = fila_mase[modelo] / naive_train_mae

    evaluacion = pd.concat([evaluacion, fila_mase], ignore_index=True)

    print(f"\nMAE del Naive sobre train (denominador MASE): {naive_train_mae:.2f}")
    print("=" * 60)
    print("  RESULTADOS POR MÉTRICA — MODELOS BASELINE")
    print("=" * 60)
    display(evaluacion)

    print("\n📌 Mejor modelo por métrica:")
    for metrica in ["mae", "rmse", "smape", "mase"]:
        fila = evaluacion[evaluacion.metric == metrica]
        mejor_idx = fila[modelos_col].values.argmin()
        mejor_modelo = modelos_col[mejor_idx]
        valor = fila[modelos_col].values[0][mejor_idx]
        print(f"   {metrica.upper():>6}: {mejor_modelo}  ({valor:.4f})")

    return evaluacion


def seleccionar_mejor_modelo(evaluacion, modelos_col, metrica="mae"):
    """Selecciona automáticamente el mejor modelo según una métrica."""
    fila = evaluacion[evaluacion.metric == metrica]
    valores = fila[modelos_col].values[0]
    mejor_idx = int(np.argmin(valores))
    return modelos_col[mejor_idx]


def graficar_metricas(evaluacion, modelos_col):
    """Grafica MAE, RMSE y sMAPE para comparar modelos."""
    fig = make_subplots(rows=1, cols=3, subplot_titles=["MAE", "RMSE", "sMAPE"])
    colores_m = ["#2196F3", "#4CAF50", "#FF5722", "#9C27B0"]

    for col_idx, metrica in enumerate(["mae", "rmse", "smape"], start=1):
        fila = evaluacion[evaluacion.metric == metrica]
        vals = [float(fila[m].values[0]) for m in modelos_col]
        mejor_idx = int(np.argmin(vals))
        bar_colors = ["gold" if i == mejor_idx else colores_m[i] for i in range(len(modelos_col))]

        fig.add_trace(
            go.Bar(
                x=modelos_col,
                y=vals,
                marker_color=bar_colors,
                name=metrica,
                showlegend=False,
                text=[f"{v:.4f}" if metrica == "smape" else f"{v:.1f}" for v in vals],
                textposition="outside"
            ),
            row=1,
            col=col_idx
        )

    fig.update_layout(
        height=400, template="plotly_white",
        title="Comparación de Métricas por Modelo<br><sup>Barra dorada = mejor modelo en esa métrica</sup>"
    )
    fig.show()

In [ ]:
def graficar_acf_plotly(serie, titulo, n_lags=30, color="#2196F3"):
    """ACF interactivo con Plotly. Barras rojas = significativas al 95%."""
    arr = np.asarray(serie).astype(float)
    arr = arr[~np.isnan(arr)]
    acf_vals = acf(arr, nlags=n_lags, fft=True)
    ci = 1.96 / np.sqrt(len(arr))
    lags = np.arange(len(acf_vals))

    fig = go.Figure()
    for lag in lags:
        bar_color = color if abs(acf_vals[lag]) <= ci else "crimson"
        fig.add_trace(go.Scatter(
            x=[lag, lag], y=[0, acf_vals[lag]], mode="lines",
            line=dict(color=bar_color, width=2.5), showlegend=False
        ))

    fig.add_trace(go.Scatter(
        x=lags, y=acf_vals, mode="markers", showlegend=False,
        marker=dict(color=[color if abs(v) <= ci else "crimson" for v in acf_vals], size=6)
    ))
    fig.add_hline(y=ci, line_dash="dash", line_color="gray", opacity=0.7, annotation_text=f"IC 95% = ±{ci:.3f}")
    fig.add_hline(y=-ci, line_dash="dash", line_color="gray", opacity=0.7)
    fig.add_hline(y=0, line_color="black", line_width=0.8)
    fig.update_layout(
        title=titulo,
        xaxis_title="Lag",
        yaxis_title="Autocorrelación",
        height=360,
        template="plotly_white",
        yaxis=dict(range=[-1.05, 1.05])
    )
    return fig


def test_estacionaridad(serie, nombre):
    """
    Aplica ADF + KPSS y entrega una conclusión conjunta.
    ADF p<0.05 + KPSS p>=0.05 → estacionaria.
    """
    arr = np.asarray(serie).astype(float)
    arr = arr[~np.isnan(arr)]

    adf_stat, adf_p, _, _, _, _ = adfuller(arr, autolag="AIC")
    kpss_stat, kpss_p, _, _ = kpss(arr, regression="c", nlags="auto")

    adf_est = adf_p < 0.05
    kpss_est = kpss_p >= 0.05

    if adf_est and kpss_est:
        conclusion = "✅  ESTACIONARIA"
    elif not adf_est and not kpss_est:
        conclusion = "❌  NO ESTACIONARIA"
    elif adf_est and not kpss_est:
        conclusion = "⚠️  INCIERTA (posible tendencia)"
    else:
        conclusion = "⚠️  INCIERTA (cerca del límite)"

    print(f"\n{'═' * 58}")
    print(f"  {nombre}")
    print(f"{'═' * 58}")
    print(f"  ADF:  stat={adf_stat:8.4f}  p={adf_p:.4f}",
          "→ ES estacionaria ✅" if adf_est else "→ NO estacionaria ❌")
    print(f"  KPSS: stat={kpss_stat:8.4f}  p={kpss_p:.4f}",
          "→ ES estacionaria ✅" if kpss_est else "→ NO estacionaria ❌")
    print(f"  {'─' * 54}")
    print(f"  CONCLUSIÓN: {conclusion}")

    return {"adf_p": adf_p, "kpss_p": kpss_p, "estacionaria": adf_est and kpss_est}


def prueba_ljungbox(serie, lags_test=None, titulo="TEST DE LJUNG-BOX"):
    """Aplica Ljung-Box para varios rezagos."""
    if lags_test is None:
        lags_test = [1, 6, 12, 18, 24]

    valores = np.asarray(serie).astype(float)
    valores = valores[~np.isnan(valores)]

    print(titulo)
    print("H₀: ρ₁ = ρ₂ = ··· = ρₕ = 0  (no hay autocorrelación)")
    print("─" * 62)
    print(f"{'Lags':>6}  {'Estadístico Q':>14}  {'p-valor':>10}  {'Conclusión'}")
    print("─" * 62)

    resultados = []
    for lag in lags_test:
        result = acorr_ljungbox(valores, lags=[lag], return_df=True)
        q_stat = result["lb_stat"].iloc[0]
        p_val = result["lb_pvalue"].iloc[0]
        conc = "Rechaza H₀ — HAY autocorrelación ❌" if p_val < 0.05 else "No rechaza H₀ — sin autocorrelación ✅"
        print(f"  {lag:>4}  {q_stat:>14.4f}  {p_val:>10.6f}  {conc}")
        resultados.append({"lag": lag, "q_stat": q_stat, "p_valor": p_val})

    print("─" * 62)
    return pd.DataFrame(resultados)


def graficar_ljungbox_pvalores(serie, max_lag=30):
    """Grafica p-valores de Ljung-Box por rezago."""
    valores = np.asarray(serie).astype(float)
    valores = valores[~np.isnan(valores)]
    lags_all = list(range(1, max_lag + 1))
    results_lb = acorr_ljungbox(valores, lags=lags_all, return_df=True)
    p_valores = results_lb["lb_pvalue"].values

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=lags_all,
        y=p_valores,
        mode="lines+markers",
        line=dict(color="#2196F3", width=2),
        marker=dict(color=["red" if p < 0.05 else "green" for p in p_valores], size=7),
        name="p-valor Ljung-Box"
    ))
    fig.add_hline(y=0.05, line_dash="dash", line_color="red", annotation_text="α = 0.05", annotation_position="right")
    fig.update_layout(
        title="Test de Ljung-Box — p-valores por lag<br><sup>Puntos rojos = evidencia de autocorrelación significativa</sup>",
        xaxis_title="Lag",
        yaxis_title="p-valor",
        height=360,
        template="plotly_white",
        yaxis=dict(range=[-0.05, 1.05])
    )
    fig.show()


def diagnosticar_residuos(test_preds, mejor_modelo, lags_residuos=None):
    """Calcula residuos del mejor modelo y aplica ACF + Ljung-Box."""
    if lags_residuos is None:
        lags_residuos = [1, 6, 12]

    residuos = test_preds["y"] - test_preds[mejor_modelo]
    residuos_arr = residuos.values

    print(f"Residuos del modelo {mejor_modelo}")
    print(f"Media: {residuos_arr.mean():.2f}  |  Desviación estándar: {residuos_arr.std():.2f}")
    print("Esperado en ruido blanco: media ≈ 0 y desviación estándar relativamente constante")

    print(f"\nLJUNG-BOX SOBRE RESIDUOS DEL {mejor_modelo}:")
    print("─" * 58)
    for lag in lags_residuos:
        result = acorr_ljungbox(residuos_arr, lags=[lag], return_df=True)
        p_val = result["lb_pvalue"].iloc[0]
        conc = "Quedan patrones ⚠️  → margen de mejora" if p_val < 0.05 else "Residuos ≈ ruido blanco ✅"
        print(f"  Lag {lag:>2}: p={p_val:.4f}  →  {conc}")

    acf_resid = acf(residuos_arr, nlags=min(52, len(residuos_arr) - 1), fft=True)
    ci_r = 1.96 / np.sqrt(len(residuos_arr))

    fig = make_subplots(rows=1, cols=2, subplot_titles=["Residuos en el tiempo", "ACF de Residuos"])
    fig.add_trace(go.Scatter(
        x=test_preds.ds, y=residuos, mode="lines+markers",
        line=dict(color="#FF5722", width=2), name="Residuos"
    ), row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)
    fig.add_hline(y=2 * residuos_arr.std(), line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_hline(y=-2 * residuos_arr.std(), line_dash="dot", line_color="gray", row=1, col=1)

    for lag_r in range(len(acf_resid)):
        color_barra = "crimson" if abs(acf_resid[lag_r]) > ci_r else "#4CAF50"
        fig.add_trace(go.Scatter(
            x=[lag_r, lag_r], y=[0, acf_resid[lag_r]],
            mode="lines", line=dict(color=color_barra, width=3), showlegend=False
        ), row=1, col=2)

    fig.add_hline(y=ci_r, line_dash="dash", line_color="gray", row=1, col=2)
    fig.add_hline(y=-ci_r, line_dash="dash", line_color="gray", row=1, col=2)
    fig.update_layout(
        height=380,
        template="plotly_white",
        showlegend=False,
        title=f"Diagnóstico de Residuos — {mejor_modelo}<br><sup>Verde = no significativo | Rojo = patrón remanente</sup>"
    )
    fig.show()
    return residuos

In [ ]:
def crear_transformaciones(serie):
    """Agrega transformaciones útiles para diagnóstico: log1p, diferencia y diferencia logarítmica."""
    transformada = serie.copy()
    transformada["y_log"] = np.log1p(transformada["y"])
    transformada["y_diff"] = transformada["y"].diff()
    transformada["y_log_diff"] = transformada["y_log"].diff()
    return transformada


def graficar_transformaciones(serie_transformada):
    """Grafica serie original y transformaciones principales."""
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=[
            "Serie original",
            "Transformación log1p(y)",
            "Diferencia logarítmica diff(log1p(y))"
        ],
        vertical_spacing=0.12
    )

    fig.add_trace(go.Scatter(
        x=serie_transformada.ds, y=serie_transformada.y,
        mode="lines", name="Original", line=dict(color="#2196F3", width=1.5)
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=serie_transformada.ds, y=serie_transformada.y_log,
        mode="lines", name="log1p(y)", line=dict(color="#4CAF50", width=1.5)
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=serie_transformada.ds, y=serie_transformada.y_log_diff,
        mode="lines", name="diff(log1p(y))", line=dict(color="#FF5722", width=1.5)
    ), row=3, col=1)

    fig.update_layout(
        height=720,
        template="plotly_white",
        title="Transformaciones de la Serie Temporal"
    )
    fig.show()


def ejecutar_validacion_cruzada(sf, serie, h=52, n_windows=3, step_size=52):
    """Ejecuta cross-validation temporal con StatsForecast."""
    cv_results = sf.cross_validation(
        df=serie,
        h=h,
        n_windows=n_windows,
        step_size=step_size
    )

    print(f"Resultados CV: {cv_results.shape[0]} filas × {cv_results.shape[1]} columnas")
    print(f"Folds (cutoffs): {cv_results.cutoff.unique().tolist()}")
    display(cv_results.head(6))
    return cv_results


def evaluar_validacion_cruzada(cv_results, modelos_col):
    """Calcula métricas promedio en la validación cruzada temporal."""
    cv_models = [c for c in modelos_col if c in cv_results.columns]
    cv_eval = evaluate(
        cv_results,
        metrics=[mae, rmse, smape],
        models=cv_models,
        target_col="y"
    )

    print("Métricas promedio en Cross-Validation:")
    print(cv_eval.to_string(index=False))
    return cv_eval


def graficar_validacion_cruzada(serie, cv_results, modelo="SeasonalNaive"):
    """Grafica los pronósticos de validación cruzada para un modelo seleccionado."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=serie.ds,
        y=serie.y,
        mode="lines",
        name="Serie real",
        line=dict(color="black", width=1.5)
    ))

    colores_cv = ["#FF5722", "#4CAF50", "#9C27B0", "#FF9800", "#2196F3"]
    for i, cutoff in enumerate(sorted(cv_results.cutoff.unique())):
        fold = cv_results[cv_results.cutoff == cutoff]
        color = colores_cv[i % len(colores_cv)]
        fig.add_trace(go.Scatter(
            x=fold.ds,
            y=fold[modelo],
            mode="lines+markers",
            name=f"{modelo} Fold {i + 1} (cutoff {cutoff.strftime('%Y-%m')})",
            line=dict(color=color, width=3, dash="dot"),
            marker=dict(size=2)
        ))
        fig.add_shape(
            type="line", xref="x", yref="paper",
            x0=cutoff, x1=cutoff, y0=0, y1=1,
            line=dict(dash="dash", color=color, width=1.5),
            opacity=0.4
        )

    fig.update_layout(
        title=f"Time Series Cross-Validation — {modelo}",
        xaxis_title="Fecha",
        yaxis_title="Casos",
        height=420,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.show()

---
## 📦 PARTE 2: Carga de Datos y Formato Nixtla

Los datos originales contienen reportes individuales de casos de dengue. Para trabajar como serie temporal, se agrupan por semana epidemiológica y se llevan al formato requerido por Nixtla:

```text
unique_id | ds | y
```

Donde `ds` es la fecha de la semana y `y` es el número de casos observados.

In [ ]:
datos_crudos = leer_datos_dengue(RUTA_DATOS_CRUDOS)

df = construir_serie_semanal_nixtla(
    datos=datos_crudos,
    columna_fecha=COLUMNA_FECHA,
    unique_id=UNIQUE_ID
)

guardar_serie_nixtla(df, RUTA_SALIDA_NIXTLA)

print("\nSerie semanal en formato Nixtla:")
display(df.head(20))
print("\nÚltimas observaciones:")
display(df.tail())

### 📈 Visualización inicial de la serie

Esta gráfica permite observar el comportamiento general de los casos de dengue a lo largo del tiempo: picos, períodos de mayor transmisión, posibles cambios de nivel y ciclos recurrentes.

In [ ]:
graficar_serie(df, titulo="Número de Casos de Dengue por Semana")

### 🔍 ¿Qué observar?

- **Picos epidémicos:** semanas con aumentos fuertes de casos.
- **Persistencia temporal:** cuando los casos altos tienden a mantenerse durante varias semanas.
- **Cambios de régimen:** períodos donde la media o la variabilidad parecen cambiar.
- **Estacionalidad:** repetición de patrones en ciertos momentos del año.

---
## 🔍 PARTE 3: Calidad de Datos

Antes de cualquier modelado, se revisa la continuidad temporal de la serie. En esta versión, primero se diagnostican faltantes y luego se completa el calendario semanal. La imputación se calcula después del split temporal para evitar usar información del test durante el entrenamiento.

### 3.1 Valores Faltantes — Detección

In [ ]:
fechas_faltantes = diagnosticar_faltantes(df, frecuencia=FRECUENCIA)

df = completar_calendario_semanal(
    serie=df,
    frecuencia=FRECUENCIA,
    unique_id=UNIQUE_ID
)

print("\nDespués de completar el calendario semanal:")
diagnosticar_faltantes(df, frecuencia=FRECUENCIA)

---
## ✂️ PARTE 4: Train-Test Split Temporal

**Regla crítica:** nunca se aleatoriza una serie temporal. El modelo debe aprender del pasado para predecir el futuro.

Se hace el corte antes de calcular las medias de imputación, de forma que cualquier regla aprendida para completar datos provenga únicamente del conjunto de entrenamiento.

In [ ]:
train, test = separar_train_test(df, FECHA_CORTE)

print(f"TRAIN: {len(train)} observaciones  ({train.ds.min().strftime('%Y-%m')} → {train.ds.max().strftime('%Y-%m')})")
print(f"TEST:  {len(test)} observaciones  ({test.ds.min().strftime('%Y-%m')} → {test.ds.max().strftime('%Y-%m')})")
print(f"Proporción: {len(train) / len(df) * 100:.0f}% / {len(test) / len(df) * 100:.0f}%")

graficar_split(train, test, FECHA_CORTE)

### 4.1 Imputación por Media Estacional

Dado que la serie puede presentar estacionalidad semanal/anual, se utiliza la media por semana del año. Para evitar fuga de información, las medias estacionales se calculan **solo con train** y luego se aplican a `train` y `test`.

In [ ]:
medias_estacionales, media_global_train = calcular_medias_estacionales(train)

train = imputar_por_media_estacional(train, medias_estacionales, media_global_train)
test = imputar_por_media_estacional(test, medias_estacionales, media_global_train)

df = pd.concat([train, test], ignore_index=True).sort_values("ds").reset_index(drop=True)

print(f"NaN en train después de imputación: {train.y.isna().sum()}")
print(f"NaN en test después de imputación:  {test.y.isna().sum()}")
print(f"NaN en df después de imputación:    {df.y.isna().sum()}")

display(df.head())
display(df.tail())

---
## 🔎 PARTE 5: Outliers y Anomalías

En una serie epidemiológica, los valores extremos no deben eliminarse automáticamente. Un pico puede ser un error de datos, pero también puede representar un brote real. Por eso, esta sección se usa como diagnóstico, no como criterio directo de eliminación.

### 5.1 Detección de Outliers — IQR y Z-score

In [ ]:
outliers_iqr, q1, q3, iqr, limite_inf_iqr, limite_sup_iqr = detectar_outliers_iqr(df)
outliers_z = detectar_outliers_zscore(df, umbral=3)

print("IQR — Detección de Outliers")
print(f"  Q1={q1:.0f}  Q3={q3:.0f}  IQR={iqr:.0f}")
print(f"  Límite inferior: {limite_inf_iqr:.0f}")
print(f"  Límite superior: {limite_sup_iqr:.0f}")
print(f"  Outliers detectados: {len(outliers_iqr)}")
display(outliers_iqr[["ds", "y"]])

print(f"\nZ-score (|z| > 3) — Outliers detectados: {len(outliers_z)}")
if len(outliers_z):
    display(outliers_z[["ds", "y"]])

graficar_outliers_iqr(df, outliers_iqr, limite_inf_iqr, limite_sup_iqr)

### 🔍 Interpretación — Outliers

Los outliers detectados por IQR no necesariamente son errores. En el caso del dengue, pueden representar brotes reales o semanas epidemiológicas con transmisión inusualmente alta. Por esta razón, se recomienda mantenerlos en la serie, salvo que exista evidencia externa de error de registro.

In [ ]:
df_check = detectar_outliers_iqr_por_mes(df)
n_out_mes = df_check.outlier_mensual.sum()

print(f"Outliers con IQR por mes: {n_out_mes}")
if n_out_mes:
    display(df_check[df_check.outlier_mensual][["ds", "y", "mes"]])
else:
    print("✅ Sin outliers cuando se aplica IQR por estación")

### 5.2 Cambios de Régimen y Anomalías Estructurales

Un cambio de régimen ocurre cuando las propiedades estadísticas de la serie cambian de forma importante: por ejemplo, la media, la varianza o la intensidad de los brotes.

In [ ]:
graficar_cambios_regimen(df, ventana=2)

In [ ]:
resultado_regimen = evaluar_cambio_regimen(df)

### 🔍 Interpretación — Cambios de Régimen

Si se observan diferencias importantes entre subperíodos, esto sugiere que la serie no se comporta igual durante todo el horizonte histórico. En dengue esto puede deberse a ciclos epidémicos, cambios climáticos, variaciones en vigilancia epidemiológica o brotes regionales.

---
## 🤖 PARTE 6: Baselines con `statsforecast`

Los modelos baseline se entrenan sobre la escala original de casos. Esto permite construir una referencia simple e interpretable antes de pasar a modelos más complejos o transformaciones.

Modelos utilizados:

- `Naive`: repite el último valor observado.
- `SeasonalNaive`: usa la misma semana del año anterior.
- `WindowAverage`: promedio de las últimas observaciones.
- `RandomWalkWithDrift`: caminata aleatoria con tendencia.

In [ ]:
sf, preds = entrenar_y_predecir_baselines(
    train=train,
    test=test,
    frecuencia=FRECUENCIA,
    estacionalidad=ESTACIONALIDAD_SEMANAL
)

test_preds = unir_predicciones_con_test(test, preds)

In [ ]:
graficar_pronosticos_baseline(
    serie=df,
    test=test,
    test_preds=test_preds,
    fecha_corte=FECHA_CORTE,
    modelos_col=MODELOS_BASELINE
)

### 🔍 ¿Qué observar en los pronósticos?

La comparación visual ayuda a identificar si los modelos baseline capturan el nivel general de la serie, si reaccionan demasiado lento ante brotes o si sobreestiman/subestiman los casos durante el período de prueba.

---
## 📊 PARTE 7: Métricas de Evaluación

Se calculan métricas de error para comparar los modelos baseline.

| Métrica | Interpretación |
|---|---|
| MAE | Error absoluto promedio en casos |
| RMSE | Penaliza más los errores grandes |
| sMAPE | Error porcentual simétrico |
| MASE | Error relativo frente a un modelo naive |

In [ ]:
evaluacion = calcular_metricas_baseline(
    train=train,
    test_preds=test_preds,
    modelos_col=MODELOS_BASELINE
)

mejor_modelo = seleccionar_mejor_modelo(
    evaluacion=evaluacion,
    modelos_col=MODELOS_BASELINE,
    metrica="mae"
)

print(f"\n✅ Mejor modelo seleccionado por MAE: {mejor_modelo}")

In [ ]:
graficar_metricas(evaluacion, MODELOS_BASELINE)

### 🔍 Conclusión de métricas

El mejor baseline debe seleccionarse con base en la métrica principal del análisis. En este notebook se usa MAE como criterio principal porque es fácil de interpretar en unidades originales: número de casos de dengue.

---
## 🔬 PARTE 8: Cierre del Ciclo — Residuos del Mejor Baseline

Después de seleccionar el mejor baseline, se analizan sus residuos. Si los residuos todavía tienen autocorrelación, significa que el modelo dejó patrones temporales sin explicar y que hay margen para modelos más avanzados.

In [ ]:
residuos = diagnosticar_residuos(
    test_preds=test_preds,
    mejor_modelo=mejor_modelo,
    lags_residuos=[1, 6, 12]
)

---
## 📈 PARTE 9: Análisis de Autocorrelación de la Serie

La ACF mide cuánto se parece la serie a sí misma en distintos rezagos. En series epidemiológicas, una autocorrelación alta suele indicar persistencia temporal: semanas con muchos casos tienden a estar cerca de otras semanas con muchos casos.

In [ ]:
fig_acf = graficar_acf_plotly(
    df.y,
    "ACF - Casos de Dengue en la ciudad de Cali",
    n_lags=30
)
fig_acf.show()

### 🔍 Interpretación — ACF

Una ACF con decaimiento lento indica dependencia temporal fuerte. Si además aparecen rezagos significativos alrededor de 52 semanas, puede existir una estructura estacional anual.

---
## 📐 PARTE 10: Tests de Estacionariedad — ADF y KPSS

Los tests ADF y KPSS ayudan a evaluar si la serie es estacionaria. En la práctica, ambos se leen de forma complementaria:

- **ADF p < 0.05:** evidencia a favor de estacionariedad.
- **KPSS p >= 0.05:** evidencia compatible con estacionariedad.

In [ ]:
r_orig = test_estacionaridad(df.y, "Casos de dengue en Cali — Serie Original")

### 🔍 Interpretación — Tests de Estacionariedad

Si la serie no es estacionaria, puede ser necesario aplicar transformaciones como logaritmo, diferencia o diferencia logarítmica antes de modelos que lo requieran, como ARIMA/SARIMA.

---
## 🔬 PARTE 11: Test de Ljung-Box sobre la Serie Original

El test de Ljung-Box evalúa si varias autocorrelaciones son simultáneamente cero. Sirve para confirmar si la serie tiene estructura temporal explotable por modelos de pronóstico.

In [ ]:
resultados_ljungbox = prueba_ljungbox(
    df.y.values,
    lags_test=[1, 6, 12, 18, 24],
    titulo="TEST DE LJUNG-BOX — Casos de Dengue Cali"
)

In [ ]:
graficar_ljungbox_pvalores(df.y.values, max_lag=30)

### 🔍 Interpretación — Ljung-Box

Si los p-valores son menores a 0.05, se rechaza la hipótesis de ausencia de autocorrelación. En ese caso, la serie no se comporta como ruido blanco y tiene estructura temporal que puede ser aprovechada por modelos de pronóstico.

---
## 🔁 PARTE 12: Transformaciones de la Serie

Las transformaciones no reemplazan a los baselines iniciales, pero ayudan a diagnosticar y preparar la serie para modelos más exigentes.

- `log1p(y)`: reduce asimetría y estabiliza varianza.
- `diff(y)`: remueve cambios de nivel o tendencia.
- `diff(log1p(y))`: aproxima cambios relativos entre semanas.

In [ ]:
df_transformado = crear_transformaciones(df)

graficar_transformaciones(df_transformado)

In [ ]:
print("Pruebas de estacionariedad sobre transformaciones:")

r_log = test_estacionaridad(
    df_transformado["y_log"].dropna(),
    "Transformación log1p(y)"
)

r_diff = test_estacionaridad(
    df_transformado["y_diff"].dropna(),
    "Diferencia de la serie diff(y)"
)

r_log_diff = test_estacionaridad(
    df_transformado["y_log_diff"].dropna(),
    "Diferencia logarítmica diff(log1p(y))"
)

### 🔍 Interpretación — Transformaciones

Si la serie original no es estacionaria, pero alguna transformación sí lo es, esa transformación puede ser útil para modelos como ARIMA/SARIMA. Sin embargo, para comparar modelos de pronóstico, las predicciones deberían volver a la escala original antes de calcular métricas finales.

---
## 🔄 PARTE 13: Time Series Cross-Validation

El split simple de train/test depende de un único corte temporal. La validación cruzada temporal evalúa los modelos en varios cortes, respetando siempre el orden del tiempo.

In [ ]:
cv_results = ejecutar_validacion_cruzada(
    sf=sf,
    serie=df,
    h=52,
    n_windows=3,
    step_size=52
)

In [ ]:
cv_eval = evaluar_validacion_cruzada(cv_results, MODELOS_BASELINE)

graficar_validacion_cruzada(
    serie=df,
    cv_results=cv_results,
    modelo=mejor_modelo if mejor_modelo in cv_results.columns else "SeasonalNaive"
)

### 🔍 Interpretación — Cross-Validation

- Cada fold usa un período diferente de entrenamiento y prueba.
- Las métricas promedio son más robustas que las de un único split.
- Si un modelo gana en el split simple pero no en validación cruzada, conviene revisar su estabilidad.

---
## 🧪 EJERCICIO 3: Ljung-Box sobre residuos de un modelo específico

Puedes usar esta celda para repetir el diagnóstico de residuos con otro modelo, por ejemplo `SeasonalNaive`.

In [ ]:
# ── EJERCICIO 3: Ljung-Box sobre residuos del SeasonalNaive ──────────────
# Cambia el nombre del modelo si quieres analizar otro baseline.

modelo_ejercicio = "SeasonalNaive"

if modelo_ejercicio in test_preds.columns:
    residuos_ejercicio = test_preds["y"] - test_preds[modelo_ejercicio]
    result_resid = acorr_ljungbox(residuos_ejercicio, lags=[1, 6, 12], return_df=True)
    print(f"Ljung-Box sobre residuos de {modelo_ejercicio}")
    display(result_resid)
else:
    print(f"El modelo {modelo_ejercicio} no está disponible en test_preds.")